# C2O Reproduction Walkthrough

This notebook is a lightweight guide through the submitted code pipeline. It does not replace the official reproduction command; it calls and documents the same code used by `make reproduce`.

Final strategy: `phase2_g5_05_expanding`, an expanding-window transparent linear ranking model using the volatility-scaled overnight-return target. Nonlinear models are included only as robustness evidence / future work, not as the submitted final strategy.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = next(p for p in candidates if (p / 'Makefile').exists() and (p / 'src' / 'c2o_strategy').exists())
os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Working directory: {ROOT}')

## Official Deliverables

The coursework requires four artefacts:

1. `01_Report/final_report.pdf`
2. `02_QuantStats/quantstats_250m.html`
3. `03_Code_Repository/` or Git URL
4. `04_Innovation_Declaration/innovation_declaration.pdf`

The code repository reproduces the report, figures, final outputs, and QuantStats tear-sheet with one command after the course-provided data files are placed in `data/`.

In [ ]:
expected_data_files = [
    'prices.parquet',
    'sp500_tr.parquet',
    'sp500_constituents.parquet',
    'sp400_constituents.parquet',
    'all_data.parquet',
    'cheapness_scores.parquet',
    'piotrosky.parquet',
    'earnings_calendar.parquet',
    'earnings_transfo.parquet',
    'short_interest_transfo.parquet',
    'gics_info.parquet',
    'regime.parquet',
    'rolling_scores_downgrade.csv',
    'rolling_scores_upgrade.csv',
]

data_dir = ROOT / 'data'
data_status = pd.DataFrame({
    'file': expected_data_files,
    'present': [(data_dir / name).exists() for name in expected_data_files],
})
data_status

## Pipeline Map

The main pipeline is intentionally modular:

- `src/c2o_strategy/data.py`: load course data, benchmark, and point-in-time panel inputs.
- `src/c2o_strategy/features.py`: construct ranked decision-time features.
- `src/c2o_strategy/alpha.py`: transparent feature-weight scoring.
- `src/c2o_strategy/portfolio.py`: portfolio construction, costs, borrow, capacity cap.
- `src/c2o_strategy/final_strategy.py`: final strategy reproduction and audit outputs.
- `src/c2o_strategy/reporting.py`: QuantStats HTML generation.
- `tools/build_corrected_report.py` and `tools/render_report_with_machine_style.py`: report source and PDF rendering.
- `tools/model_family_robustness.py`: limited model-family robustness evidence used in the report.

In [ ]:
pipeline_files = [
    'src/c2o_strategy/data.py',
    'src/c2o_strategy/features.py',
    'src/c2o_strategy/alpha.py',
    'src/c2o_strategy/portfolio.py',
    'src/c2o_strategy/final_strategy.py',
    'src/c2o_strategy/reporting.py',
    'tools/build_corrected_report.py',
    'tools/render_report_with_machine_style.py',
    'tools/model_family_robustness.py',
]
pd.DataFrame({'path': pipeline_files, 'present': [(ROOT / p).exists() for p in pipeline_files]})

## Run Full Reproduction

Set `RUN_FULL_REPRODUCTION = True` to execute the official single-command reproduction. This can take a while because it rebuilds final strategy outputs, audit evidence, model-family robustness evidence, report assets, the PDF report, and the QuantStats HTML.

In [ ]:
RUN_FULL_REPRODUCTION = False

if RUN_FULL_REPRODUCTION:
    env = os.environ.copy()
    env['PYTHON'] = '/opt/anaconda3/bin/python3'
    subprocess.run(['make', 'reproduce'], cwd=ROOT, env=env, check=True)
else:
    print('Skipped. Set RUN_FULL_REPRODUCTION = True to run: PYTHON=/opt/anaconda3/bin/python3 make reproduce')

## Inspect Final Outputs

These cells read generated outputs if they exist. If they do not exist yet, run the reproduction cell above after placing course data in `data/`.

In [ ]:
output_paths = [
    'outputs/performance_summary.csv',
    'outputs/daily_returns_50m.csv',
    'outputs/daily_returns_250m.csv',
    'outputs/daily_returns_1b.csv',
    'outputs/positions_50m.parquet',
    'outputs/positions_250m.parquet',
    'outputs/positions_1b.parquet',
    'outputs/quantstats_250m.html',
    'report/final_report.pdf',
]
pd.DataFrame({'path': output_paths, 'present': [(ROOT / p).exists() for p in output_paths]})

In [ ]:
perf_path = ROOT / 'outputs' / 'performance_summary.csv'
if perf_path.exists():
    perf = pd.read_csv(perf_path)
    display(perf)
    headline = perf.loc[perf['AUM'].eq(250_000_000.0)].iloc[0]
    print('250M headline:')
    print(f"  Net annual return: {headline['net_annual_return']:.4%}")
    print(f"  Net volatility:    {headline['net_vol']:.4%}")
    print(f"  Net Sharpe:        {headline['net_sharpe']:.3f}")
    print(f"  Max drawdown:      {headline['max_drawdown']:.2%}")
else:
    print('Missing outputs/performance_summary.csv. Run make reproduce first.')

In [ ]:
for label in ['50m', '250m', '1b']:
    path = ROOT / 'outputs' / f'daily_returns_{label}.csv'
    if path.exists():
        daily = pd.read_csv(path, parse_dates=['date'])
        print(label, 'rows=', len(daily), 'max_date=', daily['date'].max().date())
    else:
        print(label, 'missing')

## QuantStats And Report Artefacts

The QuantStats tear-sheet is generated from the 250M final strategy net returns and the `SP500_TR` benchmark. The report PDF is generated from the report source and generated figures.

In [ ]:
from IPython.display import HTML, display

links = []
for label, path in [
    ('Final report PDF', ROOT / 'report' / 'final_report.pdf'),
    ('QuantStats 250M HTML', ROOT / 'outputs' / 'quantstats_250m.html'),
    ('Innovation declaration source', ROOT / 'report' / 'innovation_declaration.md'),
]:
    if path.exists():
        links.append(f'<li><a href="{path.as_uri()}">{label}</a></li>')
    else:
        links.append(f'<li>{label}: missing</li>')

display(HTML('<ul>' + ''.join(links) + '</ul>'))

## Run Tests

Set `RUN_TESTS = True` to run the submitted test suite. In a clean checkout, the Makefile will run reproduction first if final outputs are missing.

In [ ]:
RUN_TESTS = False

if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHON'] = '/opt/anaconda3/bin/python3'
    subprocess.run(['make', 'test'], cwd=ROOT, env=env, check=True)
else:
    print('Skipped. Set RUN_TESTS = True to run: PYTHON=/opt/anaconda3/bin/python3 make test')

## Notes For Submission

- The final strategy remains the expanding-window transparent linear ranking model.
- LightGBM and HistGradientBoosting are included only as robustness evidence / future work.
- The development cutoff is `2024-12-31`; 2025-2026 is held out for marker evaluation.
- Raw coursework data is not bundled in the submission package; place it in `data/` before reproduction.